In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df_purplle = pd.read_csv(
    "Input/purplle_campaign_data_with_nulls.csv"
)

print("Purplle dataset loaded successfully!")
print("Shape:", df_purplle.shape)

Purplle dataset loaded successfully!
Shape: (55555, 16)


In [3]:
# Check missing values in Purplle dataset

missing_values = df_purplle.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (missing_values / len(df_purplle) * 100).round(2)
})

display(missing_summary[missing_summary["Missing_Count"] > 0])

,Missing_Count,Missing_Percentage
Campaign_ID,2684,4.83
Campaign_Type,2790,5.02
Target_Audience,2747,4.94
Duration,2768,4.98
Channel_Used,2744,4.94
Impressions,2721,4.90
Clicks,2712,4.88
Leads,2654,4.78
Conversions,2674,4.81
Revenue,2763,4.97


In [5]:
# Check duplicate records

print("Duplicate rows:", df_purplle.duplicated().sum())
print("Duplicate Campaign_IDs:", df_purplle["Campaign_ID"].duplicated().sum())

Duplicate rows: 0
Duplicate Campaign_IDs: 2683


In [6]:
# Handle missing Revenue and Acquisition_Cost

df_purplle["Revenue"] = df_purplle["Revenue"].fillna(0)

acquisition_cost_mean = df_purplle["Acquisition_Cost"].mean()
df_purplle["Acquisition_Cost"] = (
    df_purplle["Acquisition_Cost"].fillna(acquisition_cost_mean)
)

print("Revenue missing values:",
      df_purplle["Revenue"].isna().sum())

print("Acquisition_Cost missing values:",
      df_purplle["Acquisition_Cost"].isna().sum())

print("Acquisition_Cost mean used:",
      acquisition_cost_mean)

Revenue missing values: 0
Acquisition_Cost missing values: 0
Acquisition_Cost mean used: 377.98697920805193


In [7]:
# Fill remaining numerical columns with mean
# ROI is intentionally excluded

other_numerical_columns = [
    "Duration",
    "Impressions",
    "Clicks",
    "Leads",
    "Conversions",
    "Engagement_Score"
]

for col in other_numerical_columns:
    df_purplle[col] = df_purplle[col].fillna(
        df_purplle[col].mean()
    )

print("Remaining numerical columns handled:")
print(other_numerical_columns)

print("\nMissing values:")
display(df_purplle[other_numerical_columns].isna().sum())

Remaining numerical columns handled:
['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Engagement_Score']

Missing values:


Duration            0
Impressions         0
Clicks              0
Leads               0
Conversions         0
Engagement_Score    0
dtype: int64

In [8]:
# Fill categorical columns with mode

categorical_columns = [
    "Campaign_Type",
    "Target_Audience",
    "Channel_Used",
    "Language",
    "Customer_Segment"
]

for col in categorical_columns:
    mode_value = df_purplle[col].mode()[0]
    df_purplle[col] = df_purplle[col].fillna(mode_value)

print("Categorical columns handled using mode:")
print(categorical_columns)

print("\nMissing values:")
display(df_purplle[categorical_columns].isna().sum())

Categorical columns handled using mode:
['Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment']

Missing values:


Campaign_Type       0
Target_Audience     0
Channel_Used        0
Language            0
Customer_Segment    0
dtype: int64

In [9]:
# Convert Date column to datetime

df_purplle["Date"] = pd.to_datetime(
    df_purplle["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Date datatype:", df_purplle["Date"].dtype)
print("Missing dates after conversion:",
      df_purplle["Date"].isna().sum())

display(df_purplle[["Date"]].head())

Date datatype: datetime64[us]
Missing dates after conversion: 2651


,Date
0,2024-12-27
1,2024-11-26
2,2025-01-08
3,2025-03-14
4,2024-07-11


In [10]:
# Add Company Name

df_purplle["Company_Name"] = "Purplle"

# Calculate Profit

df_purplle["Profit"] = (
    df_purplle["Revenue"] -
    df_purplle["Acquisition_Cost"]
)

# Calculate Profit Flag

df_purplle["Profit_Flag"] = np.where(
    df_purplle["Profit"] > 0,
    "Profit",
    "Loss"
)

# Calculate ROI

df_purplle["Calculated_ROI"] = (
    df_purplle["Profit"] /
    df_purplle["Acquisition_Cost"]
)

print("Feature engineering completed successfully.")

display(
    df_purplle[
        [
            "Company_Name",
            "Revenue",
            "Acquisition_Cost",
            "Profit",
            "Profit_Flag",
            "Calculated_ROI"
        ]
    ].head(10)
)

Feature engineering completed successfully.


,Company_Name,Revenue,Acquisition_Cost,Profit,Profit_Flag,Calculated_ROI
0,Purplle,493394.0,55.93,493338.07,Profit,8820.634186
1,Purplle,108644.0,197.44,108446.56,Profit,549.263371
2,Purplle,0.0,234.59,-234.59,Loss,-1.000000
3,Purplle,1341570.0,103.76,1341466.24,Profit,12928.548959
4,Purplle,787510.0,78.61,787431.39,Profit,10016.936649
5,Purplle,158340.0,411.09,157928.91,Profit,384.171130
6,Purplle,3427100.0,60.38,3427039.62,Profit,56757.860550
7,Purplle,713856.0,42.22,713813.78,Profit,16907.005685
8,Purplle,627912.0,124.35,627787.65,Profit,5048.553679
9,Purplle,687102.0,84.20,687017.80,Profit,8159.356295


In [11]:
# Check remaining missing values

missing_after_cleaning = df_purplle.isnull().sum()

display(
    missing_after_cleaning[
        missing_after_cleaning > 0
    ]
)

Campaign_ID    2684
ROI            2731
Date           2651
dtype: int64

In [12]:
# Multi-label encoding for Channel_Used

channels = [
    "Google",
    "WhatsApp",
    "YouTube",
    "Email",
    "Instagram",
    "Facebook"
]

for channel in channels:
    df_purplle[f"Channel_{channel}"] = (
        df_purplle["Channel_Used"]
        .str.split(",")
        .apply(
            lambda x: int(
                channel in [item.strip() for item in x]
            )
        )
    )

print("Multi-label encoding completed successfully.")

display(
    df_purplle[
        [
            "Channel_Used",
            "Channel_Google",
            "Channel_WhatsApp",
            "Channel_YouTube",
            "Channel_Email",
            "Channel_Instagram",
            "Channel_Facebook"
        ]
    ].head(10)
)

Multi-label encoding completed successfully.


,Channel_Used,Channel_Google,Channel_WhatsApp,Channel_YouTube,Channel_Email,Channel_Instagram,Channel_Facebook
0,"Facebook, Google",1,0,0,0,0,1
1,"Instagram, WhatsApp",0,1,0,0,1,0
2,"Facebook, WhatsApp",0,1,0,0,0,1
3,"Instagram, WhatsApp",0,1,0,0,1,0
4,"WhatsApp, Instagram",0,1,0,0,1,0
5,Email,0,0,0,1,0,0
6,"Google, Facebook",1,0,0,0,0,1
7,"WhatsApp, Google",1,1,0,0,0,0
8,"Google, Email, Facebook",1,0,0,1,0,1
9,"Instagram, Google",1,0,0,0,1,0


In [13]:
# Validate channel indicator totals

channel_columns = [
    "Channel_Google",
    "Channel_WhatsApp",
    "Channel_YouTube",
    "Channel_Email",
    "Channel_Instagram",
    "Channel_Facebook"
]

print("Number of campaigns using each channel:")

display(
    df_purplle[channel_columns]
    .sum()
    .sort_values(ascending=False)
)

Number of campaigns using each channel:


Channel_Email        20460
Channel_Facebook     17738
Channel_YouTube      17590
Channel_Google       17583
Channel_Instagram    17513
Channel_WhatsApp     17496
dtype: int64

In [14]:
# Validate channel encoding

channel_indicator_total = (
    df_purplle[channel_columns].sum(axis=1)
)

print(
    "Campaigns with no encoded channel:",
    (channel_indicator_total == 0).sum()
)

print(
    "Campaigns with one or more encoded channels:",
    (channel_indicator_total >= 1).sum()
)

print(
    "Total campaigns:",
    len(df_purplle)
)

Campaigns with no encoded channel: 0
Campaigns with one or more encoded channels: 55555
Total campaigns: 55555


In [15]:
# Save cleaned and feature-engineered Purplle dataset

output_path = (
    r"D:\Data Science\vscode"
    r"\Marketing_Campaign_Performance_Prediction"
    r"\Output\Purplle_Feature_Engineered.csv"
)

df_purplle.to_csv(output_path, index=False)

print("Purplle dataset saved successfully.")
print("File:", output_path)
print("Shape:", df_purplle.shape)

Purplle dataset saved successfully.
File: D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\Purplle_Feature_Engineered.csv
Shape: (55555, 26)
